# YOLO26 Experimental Training and Eval


## Plain YOLO26

This section trains YOLO26 on FLoW-img dataset and tunes it for the best hyperparameters, but overall keeping defaults from Ultralytics.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26m.pt")

model.tune(
    data="data.yaml",
    epochs=50,
    patience=20,
    iterations=50,
    optimizer="MuSGD",
    save=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO("./yolo26m-FLoW-plain.pt")

model.val(
    data="data.yaml",
    save=True,
)

## SF + SAHI + YOLO26

This section is dedicated to augmenting the dataset, training YOLO26 on it and then evaluating using SAHI framework.


In [ ]:
import supervision as sv
from sahi.slicing import slice_coco

TRAIN_DIR = "FLoW-img/train"
VAL_DIR = "FLoW-img/val"


# Convert your YOLO dataset to COCO
def yolo2coco(dir):
    ds = sv.DetectionDataset.from_yolo(
        images_directory_path=f"{dir}/images",
        annotations_directory_path=f"{dir}/labels",
        data_yaml_path="data.yaml",
    )
    ds.as_coco(
        annotations_path=f"{dir}/annotations.json",
    )


# Slice your COCO dataset into smaller images with corresponding annotations
def slice_dataset(dir):
    return slice_coco(
        coco_annotation_file_path=f"{dir}/annotations.json",
        image_dir=f"{dir}/images",
        output_dir=f"{dir}_sliced/images",
        output_coco_annotation_file_name="annotations_sliced",
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
    )


# Convert the sliced COCO dataset back to YOLO format
def coco2yolo(dir):
    ds = sv.DetectionDataset.from_coco(
        annotations_path=f"{dir}_sliced/images/annotations_sliced_coco.json",
        images_directory_path=f"{dir}_sliced/images",
    )
    ds.as_yolo(
        annotations_directory_path=f"{dir}_sliced/labels",
    )


yolo2coco(TRAIN_DIR)
yolo2coco(VAL_DIR)


slice_dataset(TRAIN_DIR)
slice_dataset(VAL_DIR)

coco2yolo(TRAIN_DIR)
coco2yolo(VAL_DIR)

In [ ]:
from ultralytics import YOLO

model = YOLO("./yolo26m-FLoW-plain.pt")

model.train(
    data="data_sliced.yaml",
    epochs=100,
    patience=10,
    optimizer="MuSGD",
    project="water-plastic-detection",
    name="yolo26m-FLoW-sliced",
    cfg="best_hyperparameters.yaml",
    save=True,
)